In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum, avg, when, count, max, min, expr, stddev,
    percentile_approx, round as spark_round, desc, format_number
)

In [ ]:
spark = SparkSession.builder.getOrCreate()

In [ ]:
df_analise = spark.read.parquet('drive/MyDrive/trabalho_ebac_pyspark/02_dados_tratados/vendas_consolidadas.parquet')

In [ ]:
df_analise.printSchema()
df_analise.show(5)

root
 |-- id_transacao: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- id_produto: string (nullable = true)
 |-- data_transacao: timestamp (nullable = true)
 |-- quantidade: short (nullable = true)
 |-- preco_unitario: double (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- desconto_aplicado: short (nullable = true)
 |-- status: string (nullable = true)
 |-- metodo_pagamento: string (nullable = true)
 |-- custo_envio: double (nullable = true)
 |-- data_cadastro: date (nullable = true)
 |-- idade: short (nullable = true)
 |-- genero: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- cancelou: short (nullable = true)
 |-- valor_vitalicio: double (nullable = true)
 |-- possui_app: short (nullable = true)
 |-- categoria: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- preco_catalogo_produto: double (nullable = true)
 |-- dispositivo: string (nullable = true)
 |-- canal: stri

In [ ]:
df_analise.show(3)

+------------+----------+----------+-------------------+----------+--------------+-----------+-----------------+-----------+----------------+-----------+-------------+-----+---------+--------------+-------------------+--------+---------------+----------+-----------+----------+----------------------+----------------+----------------+-------------+----------------------+--------------------------+----------------------+----------------+---------------+--------------+--------------------+-------------+-------------------+
|id_transacao|id_cliente|id_produto|     data_transacao|quantidade|preco_unitario|valor_total|desconto_aplicado|     status|metodo_pagamento|custo_envio|data_cadastro|idade|   genero|          pais|           segmento|cancelou|valor_vitalicio|possui_app|  categoria|     marca|preco_catalogo_produto|     dispositivo|           canal|total_sessoes|media_duracao_segundos|total_paginas_visualizadas|total_adicoes_carrinho|total_conversoes|total_rejeicoes|avaliacao_nota|avalia

In [ ]:
# ==============================================================================
# BASE AUXILIAR: APENAS CONCLUÍDOS (PARA ANÁLISES QUE NÃO PRECISAM DE BRUTO)
# ==============================================================================

df_filtrado_concluido = df_analise.filter(col("status") == "concluido")

# ==============================================================================
# 2. ANÁLISE DE MÉTODOS DE PAGAMENTO (APENAS CONCLUÍDOS)
# ==============================================================================

# 2.1 Distribuição por Método de Pagamento
print("\n" + "="*60)
print("📊 DISTRIBUIÇÃO POR MÉTODO DE PAGAMENTO")
print("="*60)

# Agregação numérica para ordenação precisa (Apenas concluídos)
df_distribuicao_metodo_num = df_filtrado_concluido.groupBy("metodo_pagamento").agg(
    count("id_transacao").alias("transacoes_concluidas"),
    sum("valor_total").alias("liquido_num"),
    avg("valor_total").alias("ticket_medio_num")
).orderBy(desc("transacoes_concluidas"))

# Formatação final dos campos
df_distribuicao_metodo = df_distribuicao_metodo_num.select(
    col("metodo_pagamento"),
    col("transacoes_concluidas"),
    format_number("liquido_num", 2).alias("faturamento_liquido"),
    spark_round("ticket_medio_num", 2).alias("ticket_medio_concluido")
)

df_distribuicao_metodo.show(truncate=False)

df_distribuicao_metodo.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/trabalho_ebac_pyspark/03_resumos_estatisticos/distribuicao_metodo_pagamento'
)


📊 DISTRIBUIÇÃO POR MÉTODO DE PAGAMENTO
+----------------------+---------------------+-------------------+----------------------+
|metodo_pagamento      |transacoes_concluidas|faturamento_liquido|ticket_medio_concluido|
+----------------------+---------------------+-------------------+----------------------+
|cartao_credito        |23778                |1,881,671.47       |79.13                 |
|cartao_debito         |13993                |1,113,785.64       |79.6                  |
|paypal                |12407                |984,729.04         |79.37                 |
|apple_pay             |8195                 |667,187.49         |81.41                 |
|google_pay            |6848                 |538,069.62         |78.57                 |
|transferencia_bancaria|3479                 |257,804.78         |74.1                  |
+----------------------+---------------------+-------------------+----------------------+



In [ ]:
# ==============================================================================
# BASE AUXILIAR: APENAS CONCLUÍDOS (PARA ANÁLISES QUE NÃO PRECISAM DE BRUTO)
# ==============================================================================

df_filtrado_concluido = df_analise.filter(col("status") == "concluido")

# ==============================================================================
# 3. ANÁLISE 1: DISTRIBUIÇÕES (VISÃO BRUTO VS LÍQUIDO COM ORDENAÇÃO CORRETA)
# ==============================================================================

# 3.1 Distribuição por País
print("\n" + "="*60)
print("📊 DISTRIBUIÇÃO POR PAÍS")
print("="*60)

df_distribuicao_pais_num = df_analise.groupBy("pais").agg(
    count("id_transacao").alias("total_transacoes"),
    sum("valor_total").alias("bruto_num"),
    sum(when(col("status") == "concluido", col("valor_total")).otherwise(0)).alias("liquido_num"),
    avg(when(col("status") == "concluido", col("valor_total"))).alias("ticket_medio_num"),
    avg("custo_envio").alias("custo_envio_num")
).orderBy(desc("liquido_num"))

df_distribuicao_pais = df_distribuicao_pais_num.select(
    col("pais"),
    col("total_transacoes"),
    format_number("bruto_num", 2).alias("valor_movimentado_bruto"),
    format_number("liquido_num", 2).alias("faturamento_liquido"),
    spark_round("ticket_medio_num", 2).alias("ticket_medio_concluido"),
    spark_round("custo_envio_num", 2).alias("custo_envio_medio")
)

df_distribuicao_pais.show(truncate=False)

df_distribuicao_pais.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/trabalho_ebac_pyspark/03_resumos_estatisticos/distribuicao_pais'
)


📊 DISTRIBUIÇÃO POR PAÍS
+--------------+----------------+-----------------------+-------------------+----------------------+-----------------+
|pais          |total_transacoes|valor_movimentado_bruto|faturamento_liquido|ticket_medio_concluido|custo_envio_medio|
+--------------+----------------+-----------------------+-------------------+----------------------+-----------------+
|Estados_Unidos|47067           |3,673,352.77           |2,124,285.38       |79.0                  |4.07             |
|Reino_Unido   |14184           |1,144,604.81           |675,408.99         |82.78                 |4.03             |
|Canada        |11965           |936,059.62             |530,586.73         |77.27                 |4.07             |
|Alemanha      |9200            |725,137.89             |405,334.32         |78.19                 |4.03             |
|Franca        |7759            |601,592.61             |353,837.58         |79.23                 |4.07             |
|India         |7416   

In [ ]:
# 3.2 Distribuição por Gênero
print("\n" + "="*60)
print("📊 DISTRIBUIÇÃO POR GÊNERO")
print("="*60)

df_distribuicao_genero_num = df_analise.groupBy("genero").agg(
    count("id_transacao").alias("total_transacoes"),
    sum("valor_total").alias("bruto_num"),
    sum(when(col("status") == "concluido", col("valor_total")).otherwise(0)).alias("liquido_num"),
    avg(when(col("status") == "concluido", col("valor_total"))).alias("ticket_medio_num")
).orderBy(desc("liquido_num"))

df_distribuicao_genero = df_distribuicao_genero_num.select(
    col("genero"),
    col("total_transacoes"),
    format_number("bruto_num", 2).alias("valor_movimentado_bruto"),
    format_number("liquido_num", 2).alias("faturamento_liquido"),
    spark_round("ticket_medio_num", 2).alias("ticket_medio_concluido")
)

df_distribuicao_genero.show(truncate=False)

df_distribuicao_genero.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/trabalho_ebac_pyspark/03_resumos_estatisticos/distribuicao_genero'
)


📊 DISTRIBUIÇÃO POR GÊNERO
+-----------------+----------------+-----------------------+-------------------+----------------------+
|genero           |total_transacoes|valor_movimentado_bruto|faturamento_liquido|ticket_medio_concluido|
+-----------------+----------------+-----------------------+-------------------+----------------------+
|feminino         |54589           |4,283,088.22           |2,468,045.08       |79.06                 |
|masculino        |53672           |4,220,425.74           |2,443,956.19       |79.49                 |
|nao_binario      |5823            |465,901.32             |271,265.96         |80.66                 |
|prefiro nao dizer|5916            |444,361.57             |259,980.81         |77.05                 |
+-----------------+----------------+-----------------------+-------------------+----------------------+



In [ ]:
# 3.3 Distribuição por Segmento
print("\n" + "="*60)
print("📊 DISTRIBUIÇÃO POR SEGMENTO")
print("="*60)

df_distribuicao_segmento_num = df_analise.groupBy("segmento").agg(
    count("id_cliente").alias("total_transacoes"),
    avg("valor_vitalicio").alias("ltv_num"),
    sum("valor_total").alias("bruto_num"),
    sum(when(col("status") == "concluido", col("valor_total")).otherwise(0)).alias("liquido_num")
).orderBy(desc("liquido_num"))

df_distribuicao_segmento = df_distribuicao_segmento_num.select(
    col("segmento"),
    col("total_transacoes"),
    spark_round("ltv_num", 2).alias("media_ltv"),
    format_number("bruto_num", 2).alias("valor_movimentado_bruto"),
    format_number("liquido_num", 2).alias("faturamento_liquido")
)

df_distribuicao_segmento.show(truncate=False)

df_distribuicao_segmento.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/trabalho_ebac_pyspark/03_resumos_estatisticos/distribuicao_segmento'
)


📊 DISTRIBUIÇÃO POR SEGMENTO
+-------------------+----------------+---------+-----------------------+-------------------+
|segmento           |total_transacoes|media_ltv|valor_movimentado_bruto|faturamento_liquido|
+-------------------+----------------+---------+-----------------------+-------------------+
|Regular            |40084           |1095.18  |3,111,382.14           |1,808,827.45       |
|Premium            |33834           |2405.42  |2,663,390.76           |1,543,098.13       |
|VIP                |27794           |5056.23  |2,188,440.27           |1,253,870.60       |
|Comprador_economico|17123           |499.99   |1,359,128.91           |781,700.43         |
|Visitante_ocasional|1165            |788.3    |91,434.77              |55,751.43          |
+-------------------+----------------+---------+-----------------------+-------------------+



In [ ]:
#3.4 Distribuição por Categoria
print("\n" + "="*60)
print("📊 DISTRIBUIÇÃO POR CATEGORIA")
print("="*60)

df_distribuicao_categoria_num = df_analise.groupBy("categoria").agg(
    count("id_transacao").alias("total_vendas"),
    sum("valor_total").alias("bruto_num"),
    sum(when(col("status") == "concluido", col("valor_total")).otherwise(0)).alias("liquido_num"),
    avg("preco_unitario").alias("preco_medio_num")
).orderBy(desc("liquido_num"))

df_distribuicao_categoria = df_distribuicao_categoria_num.select(
    col("categoria"),
    col("total_vendas"),
    format_number("bruto_num", 2).alias("valor_movimentado_bruto"),
    format_number("liquido_num", 2).alias("faturamento_liquido"),
    spark_round("preco_medio_num", 2).alias("preco_medio_praticado")
)

df_distribuicao_categoria.show(truncate=False)
df_distribuicao_categoria.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/trabalho_ebac_pyspark/03_resumos_estatisticos/distribuicao_categoria'
)


📊 DISTRIBUIÇÃO POR CATEGORIA
+-------------------+------------+-----------------------+-------------------+---------------------+
|categoria          |total_vendas|valor_movimentado_bruto|faturamento_liquido|preco_medio_praticado|
+-------------------+------------+-----------------------+-------------------+---------------------+
|Eletronicos        |8796        |2,530,955.24           |1,504,492.02       |195.67               |
|Joias              |8593        |1,253,476.38           |727,779.93         |100.61               |
|Automotivo         |8289        |1,011,153.76           |573,642.18         |83.47                |
|Casa_Jardim        |8088        |728,620.00             |420,824.46         |61.11                |
|Esportes           |9050        |643,321.82             |362,736.17         |48.48                |
|Roupas             |8475        |597,954.98             |337,762.07         |48.01                |
|Saude              |10860       |579,043.09             |332

In [ ]:
# ==============================================================================
# 4. ANÁLISE 2: MÉDIA E MEDIANA POR SEGMENTO
# ==============================================================================

print("\n" + "="*60)
print("📊 MÉDIA E MEDIANA POR SEGMENTO")
print("="*60)

df_media_mediana = df_analise.groupBy("segmento").agg(
    spark_round(avg("media_duracao_segundos"), 0).alias("media_duracao_sessao_seg"),
    spark_round(avg("total_paginas_visualizadas"), 0).alias("media_paginas_vistas"),
    spark_round(avg(when(col("status") == "concluido", col("valor_total"))), 2).alias("ticket_medio_concluido"),
    spark_round(expr("percentile_approx(valor_vitalicio, 0.5)"), 2).alias("mediana_valor_vitalicio"),
    spark_round(avg("valor_vitalicio"), 2).alias("media_valor_vitalicio")
).orderBy(desc("ticket_medio_concluido"))

df_media_mediana.show(truncate=False)


df_media_mediana.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/trabalho_ebac_pyspark/03_resumos_estatisticos/media_mediana_segmento'
)


📊 MÉDIA E MEDIANA POR SEGMENTO
+-------------------+------------------------+--------------------+----------------------+-----------------------+---------------------+
|segmento           |media_duracao_sessao_seg|media_paginas_vistas|ticket_medio_concluido|mediana_valor_vitalicio|media_valor_vitalicio|
+-------------------+------------------------+--------------------+----------------------+-----------------------+---------------------+
|Visitante_ocasional|89.0                    |3.0                 |84.22                 |784.41                 |788.3                |
|Comprador_economico|168.0                   |11.0                |79.94                 |500.93                 |499.99               |
|Premium            |405.0                   |82.0                |79.72                 |2399.03                |2405.42              |
|VIP                |578.0                   |193.0               |79.45                 |5033.2                 |5056.23              |
|Regular 

In [ ]:
# ==============================================================================
# 5. ANÁLISE 3: OUTLIERS (CLIENTES COM ALTO LTV)
# ==============================================================================

print("\n" + "="*60)
print("📊 OUTLIERS - CLIENTES COM ALTO VALOR VITALÍCIO")
print("="*60)

stats_ltv = df_analise.select(
    avg("valor_vitalicio").alias("media"),
    stddev("valor_vitalicio").alias("desvio")
).collect()

media_ltv = stats_ltv[0]["media"]
desvio_ltv = stats_ltv[0]["desvio"]
limite_superior = media_ltv + (3 * desvio_ltv)

df_outliers = df_analise.filter(col("valor_vitalicio") > limite_superior) \
                        .select("id_cliente", "segmento", "pais", "idade", "valor_vitalicio") \
                        .distinct() \
                        .orderBy(desc("valor_vitalicio"))

print(f"🔍 Limite para Outlier (+3 Desvios Padrão): R$ {limite_superior:.2f}")
print(f"📌 Total de clientes outlier: {df_outliers.count()}")
df_outliers.show(truncate=False)
df_outliers.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/trabalho_ebac_pyspark/03_resumos_estatisticos/outliers_ltv'
)



📊 OUTLIERS - CLIENTES COM ALTO VALOR VITALÍCIO
🔍 Limite para Outlier (+3 Desvios Padrão): R$ 7672.10
📌 Total de clientes outlier: 20
+----------+--------+--------------+-----+---------------+
|id_cliente|segmento|pais          |idade|valor_vitalicio|
+----------+--------+--------------+-----+---------------+
|C04587    |VIP     |Estados_Unidos|33   |9353.25        |
|C08939    |VIP     |Reino_Unido   |48   |9320.91        |
|C03808    |VIP     |Estados_Unidos|20   |9026.7         |
|C07156    |VIP     |Canada        |33   |9007.12        |
|C09668    |VIP     |Franca        |59   |8990.45        |
|C00584    |VIP     |Alemanha      |55   |8580.5         |
|C03714    |VIP     |Australia     |42   |8340.74        |
|C06340    |VIP     |Estados_Unidos|26   |8297.84        |
|C06246    |VIP     |Alemanha      |18   |8247.14        |
|C07743    |VIP     |Mexico        |26   |8174.02        |
|C07284    |VIP     |Estados_Unidos|39   |8122.57        |
|C00936    |VIP     |Estados_Unidos|49  

In [ ]:
# ==============================================================================
# 6. ANÁLISE 4: CORRELAÇÕES
# ==============================================================================

print("\n" + "="*60)
print("📊 CORRELAÇÕES ENTRE VARIÁVEIS")
print("="*60)

corr_idade_ltv = df_analise.stat.corr("idade", "valor_vitalicio")
corr_idade_valor = df_analise.stat.corr("idade", "valor_total")
corr_paginas_carrinho = df_analise.stat.corr("total_paginas_visualizadas", "total_adicoes_carrinho")
corr_duracao_valor = df_analise.stat.corr("media_duracao_segundos", "valor_total")
corr_preco_quantidade = df_analise.stat.corr("preco_unitario", "quantidade")
corr_desconto_valor = df_analise.stat.corr("desconto_aplicado", "valor_total")
corr_ltv_cancelou = df_analise.stat.corr("valor_vitalicio", "cancelou")

dados_correlacao = [
    ("Idade", "Valor Vitalício", corr_idade_ltv),
    ("Idade", "Valor Total", corr_idade_valor),
    ("Páginas Vistas", "Adições Carrinho", corr_paginas_carrinho),
    ("Duração Sessão", "Valor Total", corr_duracao_valor),
    ("Preço Unitário", "Quantidade", corr_preco_quantidade),
    ("Desconto Aplicado", "Valor Total", corr_desconto_valor),
    ("LTV", "Cancelou", corr_ltv_cancelou)
]

df_correlacao = spark.createDataFrame(
    dados_correlacao,
    ["variavel_1", "variavel_2", "coeficiente"]
)

df_correlacao.show(truncate=False)
df_correlacao.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/trabalho_ebac_pyspark/03_resumos_estatisticos/correlacao_variaveis'
)


📊 CORRELAÇÕES ENTRE VARIÁVEIS
+-----------------+----------------+----------------------+
|variavel_1       |variavel_2      |coeficiente           |
+-----------------+----------------+----------------------+
|Idade            |Valor Vitalício |-3.960755684081463E-4 |
|Idade            |Valor Total     |-0.005708041109254804 |
|Páginas Vistas   |Adições Carrinho|0.7836384428334142    |
|Duração Sessão   |Valor Total     |-0.0033513446699613156|
|Preço Unitário   |Quantidade      |-0.00246291545264241  |
|Desconto Aplicado|Valor Total     |-0.09130543630048266  |
|LTV              |Cancelou        |-0.199056259173722    |
+-----------------+----------------+----------------------+



In [ ]:
# ==============================================================================
# 7. ANÁLISE 5: DISPERSÃO E CRUZAMENTOS ESTRATÉGICOS
# ==============================================================================

# 7.1 Dispersão por Categoria e Canal
print("\n" + "="*60)
print("📊 DISPERSÃO - VENDAS POR CATEGORIA E CANAL")
print("="*60)

df_dispersao_num = df_analise.groupBy("categoria", "canal").agg(
    count("id_transacao").alias("quantidade_vendas"),
    sum("valor_total").alias("bruto_num"),
    sum(when(col("status") == "concluido", col("valor_total")).otherwise(0)).alias("liquido_num"),
    avg(when(col("status") == "concluido", col("valor_total"))).alias("ticket_medio_num"),
    avg("preco_unitario").alias("preco_medio_num")
).orderBy(desc("liquido_num"))

df_dispersao = df_dispersao_num.select(
    col("categoria"),
    col("canal"),
    col("quantidade_vendas"),
    format_number("bruto_num", 2).alias("valor_movimentado_bruto"),
    format_number("liquido_num", 2).alias("faturamento_liquido"),
    spark_round("ticket_medio_num", 2).alias("ticket_medio_concluido"),
    spark_round("preco_medio_num", 2).alias("preco_medio")
)

df_dispersao.show(20, truncate=False)

df_dispersao.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/trabalho_ebac_pyspark/03_resumos_estatisticos/dispersao_categoria_canal'
)

# 7.2 Dispersão por Mês e Status (saúde operacional - todos os status)
print("\n" + "="*60)
print("📊 DISPERSÃO - VENDAS POR MÊS E STATUS (TODOS OS STATUS)")
print("="*60)

df_dispersao_mes_status_num = df_analise.groupBy("mes_transacao", "status").agg(
    count("id_transacao").alias("total_pedidos"),
    sum("valor_total").alias("faturamento_num"),
    avg("valor_total").alias("ticket_medio_num")
).orderBy("mes_transacao", "status")

df_dispersao_mes_status = df_dispersao_mes_status_num.select(
    col("mes_transacao"),
    col("status"),
    col("total_pedidos"),
    format_number("faturamento_num", 2).alias("faturamento"),
    spark_round("ticket_medio_num", 2).alias("ticket_medio")
)

df_dispersao_mes_status.show(20, truncate=False)

df_dispersao_mes_status.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/trabalho_ebac_pyspark/03_resumos_estatisticos/dispersao_mes_status'
)

# 7.3 Dispersão por Faixa de Preço e Status (visão geral por status)
print("\n" + "="*60)
print("📊 DISPERSÃO - VENDAS POR FAIXA DE PREÇO E STATUS")
print("="*60)

df_dispersao_faixa_status_num = df_analise.groupBy("faixa_preco_produto", "status").agg(
    count("id_transacao").alias("total_vendas"),
    sum("valor_total").alias("faturamento_num"),
    avg("valor_total").alias("ticket_medio_num")
).orderBy(desc("faturamento_num"))

df_dispersao_faixa_status = df_dispersao_faixa_status_num.select(
    col("faixa_preco_produto"),
    col("status"),
    col("total_vendas"),
    format_number("faturamento_num", 2).alias("faturamento"),
    spark_round("ticket_medio_num", 2).alias("ticket_medio")
)

df_dispersao_faixa_status.show(truncate=False)

df_dispersao_faixa_status.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    '/content/drive/MyDrive/trabalho_ebac_pyspark/03_resumos_estatisticos/dispersao_faixa_status'
)

print("\n" + "="*60)
print("✅ Análise do Script concluída com ordenação correta e visão analítica Bruto vs Líquido!")
print("📁 Arquivos estatísticos atualizados em: 03_resumos_estatisticos/")
print("="*60)


📊 DISPERSÃO - VENDAS POR CATEGORIA E CANAL
+-------------------+----------------+-----------------+-----------------------+-------------------+----------------------+-----------+
|categoria          |canal           |quantidade_vendas|valor_movimentado_bruto|faturamento_liquido|ticket_medio_concluido|preco_medio|
+-------------------+----------------+-----------------+-----------------------+-------------------+----------------------+-----------+
|Eletronicos        |nao_identificado|8796             |2,530,955.24           |1,504,492.02       |291.0                 |195.67     |
|Joias              |nao_identificado|8590             |1,252,583.72           |726,939.15         |145.36                |100.61     |
|Automotivo         |nao_identificado|8288             |1,011,122.68           |573,611.10         |123.17                |83.47      |
|Casa_Jardim        |nao_identificado|8087             |728,549.55             |420,824.46         |90.46                 |61.1       |
|Esp